In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 156 (delta 58), reused 132 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 19.98 MiB | 17.09 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [7]:
import os
import time
import jax
import jax.numpy as jnp

from core import Pars2d, Equation2d
from solver import Solver2d, FastSolver2d
from boundaries import periodic_bc_2d
from equations import make_euler_explosion_2d

In [9]:
def flux_x(u): return u
def flux_y(u): return u
def spec_x(u): return jnp.ones_like(u)
def spec_y(u): return jnp.ones_like(u)
def initial_data(x, y):
    # Двумерный синус (перенос)
    return jnp.sin(2 * jnp.pi * x) * jnp.sin(2 * jnp.pi * y)

def run_gpu_benchmark():
    # Проверка, что Colab использует GPU
    print(f"JAX Device(s): {jax.devices()}")

    J = 200
    pars = Pars2d(
        x_init=0.0, x_final=1.0,
        y_init=0.0, y_final=1.0,
        t_final=0.5, dt_out=0.5,
        Jx=J, Jy=J,
        cfl=0.45, scheme="sd2"
    )

    # eqn = Equation2d(
    #     flux_x=flux_x, flux_y=flux_y,
    #     spectral_radius_x=spec_x, spectral_radius_y=spec_y,
    #     initial_data=initial_data,
    #     boundary_handler=periodic_bc_2d,
    #     name="Linear Advection 2D"
    # )

    eqn = make_euler_explosion_2d()

    # Используем обычный Solver2d, чтобы избежать зависания компиляции XLA на больших сетках!
    solver = FastSolver2d(pars, eqn, limiter_name="mc")

    print(f"--- Запуск JAX GPU Бенчмарка (Сетка: {J}x{J}) ---")
    # Засекаем чистое время решателя
    t0 = time.time()
    results = solver.solve()

    # Синхронизация JAX (ожидание завершения асинхронных вычислений на GPU)
    results['u'][-1].block_until_ready()
    t1 = time.time()

    print(f"\n[GPU JAX] Полное время выполнения: {t1 - t0:.4f} секунд")

if __name__ == "__main__":
    # Скрываем предупреждения JAX
    jax.config.update("jax_enable_x64", True)
    run_gpu_benchmark()

JAX Device(s): [CudaDevice(id=0)]
--- Запуск JAX GPU Бенчмарка (Сетка: 200x200) ---
Starting 2D simulation: Euler 2D (Explosion)
Grid: 200x200, Scheme: SD2/monotonized_central
t = 0.5000 / 0.5000
Done! Wall time: 4.89 s

[GPU JAX] Полное время выполнения: 5.6857 секунд
